# Farm Copilot — pepper classification baseline

This notebook is a launcher for the standalone training package. It does not download AI Hub data or upload local files. Confirm AI Hub data-use terms before placing any derived images in Drive. A one-class technical subset is not eligible for model training.

In [ ]:
import platform
import shutil
import sys
from pathlib import Path

import torch
import torchvision

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Colab local disk:', shutil.disk_usage('/content'))
RUN_TRAINING = False  # Set True only after 3-class data/splits pass validation.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/farm-copilot-ai')
TRAINING_PACKAGE = PROJECT / 'code' / 'model_training'
DATA_ROOT = PROJECT / 'datasets' / 'pepper-v1'
LOCAL_DATA_ROOT = Path('/content/farm-copilot-ai/datasets/pepper-v1')
MANIFEST = DATA_ROOT / 'manifest.csv'
OUTPUT = PROJECT / 'experiments' / 'baseline' / 'run-001'
sys.path.insert(0, str(TRAINING_PACKAGE.parent))
print('Package:', TRAINING_PACKAGE, TRAINING_PACKAGE.exists())
print('Dataset:', DATA_ROOT, DATA_ROOT.exists())
if DATA_ROOT.exists():
    data_bytes = sum(path.stat().st_size for path in DATA_ROOT.rglob('*') if path.is_file())
    print('Dataset bytes:', data_bytes)
    print('Drive free bytes:', shutil.disk_usage(PROJECT).free)
    print('Colab local free bytes:', shutil.disk_usage('/content').free)


In [ ]:
if RUN_TRAINING:
    from model_training.train_classifier import train

    if not torch.cuda.is_available():
        print('No GPU assigned. This run will use CPU; consider stopping if that is not intended.')
    if not DATA_ROOT.is_dir() or not MANIFEST.is_file():
        raise FileNotFoundError('Approved prepared dataset and manifest are required.')
    if LOCAL_DATA_ROOT.exists():
        raise FileExistsError(f'Colab staging path already exists: {LOCAL_DATA_ROOT}')
    required_bytes = sum(path.stat().st_size for path in DATA_ROOT.rglob('*') if path.is_file())
    if required_bytes > shutil.disk_usage('/content').free:
        raise OSError('Insufficient Colab runtime disk space to stage the dataset.')
    shutil.copytree(DATA_ROOT, LOCAL_DATA_ROOT)
    local_manifest = LOCAL_DATA_ROOT / 'manifest.csv'
    result = train(
        local_manifest,
        LOCAL_DATA_ROOT,
        OUTPUT,
        epochs=5,
        batch_size=16,
        image_size=224,
        seed=2026,
        pretrained=True,
        device_name='auto',
    )
    result
else:
    print('Preflight only: set RUN_TRAINING=True after a valid 3-class manifest is ready.')
